# Evaluation on pursuit-evasion: empirical exploitability (Table 2, PE column)
Every frozen pursuer plays the same 200 pre-drawn episodes (layout, exit, and initial positions) against four evaders:

1. `Trained P2`, the trained physical evader,
2. `Critic-BR P2`, a myopic best response to the pursuer's current mixed move under the critic,
3. `Equilibrium P2`, the equilibrium column of the critic's stage game, and
4. `BR-PPO P2`, a recurrent-PPO best response trained from scratch against that pursuer (`best_response.ipynb`).

Returns are the pursuer's discounted payoff (γ = 0.99, at most 40 rounds), so evaders minimize them. A pursuer's worst-case return $\widehat W$ is its lowest mean over the evaders, and its empirical exploitability is $\widehat\Delta = \widehat V^\star - \widehat W$, where $\widehat V^\star$ is the best worst-case return among the evaluated pursuers. Every pursuer samples its move from its own distribution, as at deployment.

In [ ]:
%run Initialize.ipynb

## Settings, trained models, and the shared episodes

In [ ]:
GAMMA = 0.99
num_episodes = 200
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(DefenderActor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)
Player2       = Load(AttackerActor().to(device), "player2.pth")

episode_configs = []
cfg_rng = np.random.RandomState(12345)
for _ in range(num_episodes):
    layout_idx = cfg_rng.randint(0, len(LAYOUT_BANK))
    Epi_Layout = set(LAYOUT_BANK[layout_idx])
    exit_corner = cfg_rng.randint(0, 4)
    exit_rc = EXITS[exit_corner]
    p_row = cfg_rng.randint(0, 6); p_col = cfg_rng.randint(0, 6)
    while (p_row, p_col) in Epi_Layout:
        p_row = cfg_rng.randint(0, 6); p_col = cfg_rng.randint(0, 6)
    e_row = cfg_rng.randint(0, 6); e_col = cfg_rng.randint(0, 6)
    while (e_row, e_col) in Epi_Layout or (e_row, e_col) == (p_row, p_col) or (e_row, e_col) == exit_rc:
        e_row = cfg_rng.randint(0, 6); e_col = cfg_rng.randint(0, 6)
    episode_configs.append((layout_idx, exit_corner, p_row, p_col, e_row, e_col))

## Evaders (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
BR_CACHE = {}
def _load_br(def_name):
    if def_name not in BR_CACHE:
        ck = torch.load(os.path.join(SAVE_DIR, "BR_attacker_" + def_name.replace(" ", "_") + ".pth"),
                        map_location=device, weights_only=True)
        att = BRAttacker(rnn_h=ck['config']['RNN_H']).to(device)
        att.load_state_dict(ck['attacker']); att.eval()
        BR_CACHE[def_name] = att
    return BR_CACHE[def_name]


class BRAttackerWrapper:
    def __init__(self, def_name):
        self.att = _load_br(def_name)
    def reset(self, Theta):
        self.h = None
        self.u_prev = np.zeros(n_u, dtype=np.float32)
        self.theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
    def act(self, x):
        obs = np.concatenate([x, self.u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, _, self.h = self.att(torch.tensor(obs, device=device).view(1, 1, 2 * d + n_u), self.theta_t, self.h)
        u = int(Categorical(logits=logits.view(-1)).sample())
        self.u_prev = np.zeros(n_u, dtype=np.float32); self.u_prev[u] = 1.0
        return u


def attacker_action(Attacker, a_dist, critic_Q, x_t, m2_t, tt_t):
    # critic_Q() returns the pursuer's payoff matrix under the critic, computed only when needed
    if Attacker == 'Trained P2':
        with torch.no_grad():
            u_dist = Player2(x_t, m2_t, tt_t).cpu().numpy()
        return int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
    Q = critic_Q()
    if Attacker == 'Critic-BR P2':
        return int(np.argmin(a_dist @ Q))            # minimizes the pursuer's expected payoff
    if Attacker == 'Equilibrium P2':
        _, _, q = MatrixValue(Q)
        q = np.clip(q, 0, None); q = q / q.sum()
        return int(np.random.choice(n_u, p=q))
    raise ValueError(Attacker)

## Rollout

In [ ]:
def run_episodes(Defender, Attacker):
    defw = get_defender(Defender)
    br = BRAttackerWrapper(Defender) if Attacker == 'BR-PPO P2' else None
    ep_disc = []
    for i_episode in tqdm(range(num_episodes), desc=f"{Defender} vs {Attacker}", leave=False):
        np.random.seed(1000 + i_episode); torch.manual_seed(1000 + i_episode); random.seed(1000 + i_episode)
        layout_idx, exit_corner, p_row, p_col, e_row, e_col = episode_configs[i_episode]
        Epi_Layout = set(LAYOUT_BANK[layout_idx])
        exit_rc = EXITS[exit_corner]
        Theta, Type_Theta = theta_of(Epi_Layout, exit_corner)
        defw.reset(Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col)
        if br is not None:
            br.reset(Theta)
        # The critic's view of the episode, used by the critic-based evaders
        z_s = init_z_s(p_row, p_col)
        Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
        m2 = init_m2(e_row, e_col)
        th_t = torch.tensor(Theta, dtype=torch.float32, device=device)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        disc_sum = 0.0
        for t in range(max_steps):
            x = build_x(p_row, p_col, e_row, e_col)
            x_t = torch.tensor(x, dtype=torch.float32, device=device)
            m2_t = torch.tensor(m2, dtype=torch.float32, device=device)

            def critic_Q():
                ctx = torch.tensor(Context, dtype=torch.float32, device=device).view(1, max_steps, CTX_DIM)
                with torch.no_grad():
                    Emb = Context_Model(ctx, Context_Mask(t)).squeeze(0)
                    return critic(torch.tensor(z_s, dtype=torch.float32, device=device), Emb, x_t, m2_t,
                                  tt_t, th_t).cpu().numpy().reshape(n_a, n_u)

            a_dist = defw.act_dist(x, p_row, p_col)
            u = br.act(x) if br is not None else attacker_action(Attacker, a_dist, critic_Q, x_t, m2_t, tt_t)
            a = int(np.random.choice(n_a, p=a_dist))         # the pursuer samples from its own policy
            new_p_row, new_p_col, new_e_row, new_e_col = next_state(p_row, p_col, e_row, e_col, a, u, Epi_Layout)
            Captured = is_capture((p_row, p_col), (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col))
            timeout = (t == max_steps - 1)
            r = Reward(new_p_row, new_p_col, new_e_row, new_e_col, exit_rc, timeout, Captured)
            disc_sum += (GAMMA ** t) * r
            next_m2 = m2_update(m2, e_row, e_col, u, new_e_row, new_e_col)
            y = Observation(new_e_row, new_e_col)
            defw.observe(a, u, (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col), y)
            next_z_s = Def_State_Blf(z_s, y, new_p_row, new_p_col)
            Context[t, :] = Context_Row(z_s, a, next_z_s)
            if Captured or timeout:
                break
            z_s, m2 = next_z_s, next_m2
            p_row, p_col, e_row, e_col = new_p_row, new_p_col, new_e_row, new_e_col
        ep_disc.append(disc_sum)
    return np.asarray(ep_disc)

## Evaluate every pursuer against every evader

In [ ]:
ATTACKERS = ['Trained P2', 'Critic-BR P2', 'Equilibrium P2', 'BR-PPO P2']
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']

all_runs = {}
for Defender in DEFENDERS:
    all_runs[Defender] = {Attacker: run_episodes(Defender, Attacker) for Attacker in ATTACKERS}
    print(Defender, {k: round(v.mean(), 3) for k, v in all_runs[Defender].items()})

## Worst-case return and empirical exploitability

In [ ]:
worst_runs = {}
for Defender in DEFENDERS:
    runs = all_runs[Defender]
    worst = min(runs, key=lambda k: runs[k].mean())          # evaders minimize the pursuer's return
    worst_runs[Defender] = (worst, runs[worst])

anchor = max(worst_runs, key=lambda k: worst_runs[k][1].mean())
V_star = worst_runs[anchor][1]

print(f"anchor pursuer (least exploitable): {anchor}")
print(f"{'Pursuer':<12}{'worst evader':>16}{'W':>9}{'Delta':>9}{'SE':>8}")
print("-" * 54)
for Defender in DEFENDERS:
    worst, W = worst_runs[Defender]
    Delta = V_star - W                                         # paired over the shared episodes
    print(f"{Defender:<12}{worst:>16}{W.mean():>9.3f}{Delta.mean():>9.3f}{Delta.std(ddof=1) / np.sqrt(len(Delta)):>8.3f}")